# Data Wrangling: Join, Combine, 

In [1]:
import numpy as np
import pandas as pd
pd.options.display.max_rows = 20
np.random.seed(12345)
import matplotlib.pyplot as plt
plt.rc('figure', figsize=(10, 6))
np.set_printoptions(precision=4, suppress=True)

## Hierarchical Indexing

In [2]:
data = pd.Series(np.random.randn(9),
                 index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])
data

a  1   -0.204708
   2    0.478943
   3   -0.519439
b  1   -0.555730
   3    1.965781
c  1    1.393406
   2    0.092908
d  2    0.281746
   3    0.769023
dtype: float64

The “gaps” in the index display mean “use the label directly above”.

In [3]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

In [4]:
# จากในหนังสือ:
# MultiIndex(levels=[['a', 'b', 'c', 'd'], [1, 2, 3]],
# labels=[[0, 0, 0, 1, 1, 2, 2, 3, 3], [0, 1, 2, 0, 2, 0, 1, 1, 2]])

In [5]:
np.array([1, 2, 3])[[0, 1, 2, 0, 2, 0, 1, 1, 2]]
# สรุปได้ว่า ไอ้ list ยาวๆ มันเป็น index ของ hierachical index ตัวที่สอง ใน array [1,2,3] 

array([1, 2, 3, 1, 3, 1, 2, 2, 3])

In [6]:
print(data['b'])
print(data[['b']])
print(data['b':'c'])
print(data.loc[['b', 'd']])

1   -0.555730
3    1.965781
dtype: float64
b  1   -0.555730
   3    1.965781
dtype: float64
b  1   -0.555730
   3    1.965781
c  1    1.393406
   2    0.092908
dtype: float64
b  1   -0.555730
   3    1.965781
d  2    0.281746
   3    0.769023
dtype: float64


In [7]:
#ลองเอง
print(data[['b', 'd']])

b  1   -0.555730
   3    1.965781
d  2    0.281746
   3    0.769023
dtype: float64


In [8]:
# Selection from an “inner” level:
data.loc[:, 2]

a    0.478943
c    0.092908
d    0.281746
dtype: float64

Hierarchical indexing plays an important role in reshaping data and group-based
operations like forming a pivot table. For example, you could rearrange the data into
a DataFrame using its `unstack` method:

In [9]:
data.unstack()

,1,2,3
a,-0.204708,0.478943,-0.519439
b,-0.555730,NaN,1.965781
c,1.393406,0.092908,NaN
d,NaN,0.281746,0.769023


The inverse operation of unstack is stack:

In [10]:
data.unstack().stack()

a  1   -0.204708
   2    0.478943
   3   -0.519439
b  1   -0.555730
   3    1.965781
c  1    1.393406
   2    0.092908
d  2    0.281746
   3    0.769023
dtype: float64

In [11]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
                     columns=[['Ohio', 'Ohio', 'Colorado'],
                              ['Green', 'Red', 'Green']])
frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

In [12]:
frame.index.names = ['key1', 'key2']
frame.columns.names = ['state', 'color']
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [13]:
print(frame[['Ohio']])
frame['Ohio']

state      Ohio    
color     Green Red
key1 key2          
a    1        0   1
     2        3   4
b    1        6   7
     2        9  10


color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

In [14]:
pd.MultiIndex.from_arrays([['Ohio', 'Ohio', 'Colorado'], ['Green', 'Red', 'Green']],
                       names=['state', 'color'])

MultiIndex([(    'Ohio', 'Green'),
            (    'Ohio',   'Red'),
            ('Colorado', 'Green')],
           names=['state', 'color'])

In [15]:
from pandas import MultiIndex as mi
mi.from_arrays([['Ohio', 'Ohio', 'Colorado'], ['Green', 'Red', 'Green']],
                       names=['state', 'color'])

MultiIndex([(    'Ohio', 'Green'),
            (    'Ohio',   'Red'),
            ('Colorado', 'Green')],
           names=['state', 'color'])

### Reordering and Sorting Levels

In [16]:
frame.swaplevel('key1', 'key2')
# "gap" ไม่ใช้ในกรณีของ inner level

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

In [17]:
print(frame.sort_index(level=1)) # sort inner level (key2)
frame.swaplevel(0, 1).sort_index(level=0) # sort outer level 
# สังเกตว่าถ้า outer level (key2) is sorted มันจะมี gap ให้เห็น (เพราะเวลาถูก sort ค่าที่ tie จะติดกัน)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11


state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

![](chan_folder/hier_outermost.png)

In [18]:
#ลองเอง
print(frame, '\n')
print(frame.sort_index(level=0, axis = 1))

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11 

state     Colorado  Ohio    
color        Green Green Red
key1 key2                   
a    1           2     0   1
     2           5     3   4
b    1           8     6   7
     2          11     9  10


### Summary Statistics by Level

In [19]:
frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

In [20]:
print(frame.sum(level='key2'), '\n')
print(frame.sum(level='key1', axis=0), '\n')
print(frame.sum(level='color', axis=1)) # performed across columns belonging to the same group (not every columns) 

state  Ohio     Colorado
color Green Red    Green
key2                    
1         6   8       10
2        12  14       16 

state  Ohio     Colorado
color Green Red    Green
key1                    
a         3   5        7
b        15  17       19 

color      Green  Red
key1 key2            
a    1         2    1
     2         8    4
b    1        14    7
     2        20   10


### Indexing with a DataFrame's columns

It’s not unusual to want to use one or more columns from a DataFrame as the row
index; alternatively, you may wish to move the row index into the DataFrame’s columns.

In [21]:
frame_ = pd.DataFrame({'a': range(7), 'b': range(7, 0, -1),
                      'c': ['one', 'one', 'one', 'two', 'two',
                            'two', 'two'],
                      'd': [0, 1, 2, 0, 1, 2, 3]})
frame_

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


In [22]:
frame2 = frame_.set_index(['c', 'd'])
frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

In [23]:
frame_.set_index(['c', 'd'], drop=False)

a  b    c  d
c   d              
one 0  0  7  one  0
    1  1  6  one  1
    2  2  5  one  2
two 0  3  4  two  0
    1  4  3  two  1
    2  5  2  two  2
    3  6  1  two  3

In [24]:
frame2.reset_index()

,c,d,a,b
0,one,0,0,7
1,one,1,1,6
2,one,2,2,5
3,two,0,3,4
4,two,1,4,3
5,two,2,5,2
6,two,3,6,1


In [25]:
#ลองเอง
print(frame)
print()

myframe = frame.reset_index()
print(myframe)
myframe.columns

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

state key1 key2  Ohio     Colorado
color           Green Red    Green
0        a    1     0   1        2
1        a    2     3   4        5
2        b    1     6   7        8
3        b    2     9  10       11


MultiIndex([(    'key1',      ''),
            (    'key2',      ''),
            (    'Ohio', 'Green'),
            (    'Ohio',   'Red'),
            ('Colorado', 'Green')],
           names=['state', 'color'])

## Combining and Merging Datasets


- pandas.merge connects rows in DataFrames based on one or more keys. This
will be familiar to users of SQL or other relational databases, as it implements
database join operations.
- pandas.concat concatenates or “stacks” together objects along an axis.
- The combine_first instance method enables splicing together overlapping data
to fill in missing values in one object with values from another.

### Database-Style DataFrame Joins

In [26]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],
                    'data1': range(7)})
df2 = pd.DataFrame({'key': ['a', 'b', 'd'],
                    'data2': range(3)})
print(df1)
print(df2)

  key  data1
0   b      0
1   b      1
2   a      2
3   c      3
4   a      4
5   a      5
6   b      6
  key  data2
0   a      0
1   b      1
2   d      2


This is an example of **a many-to-one join**; the data in `df1` has multiple rows labeled `a`
and `b`, whereas `df2` has only one row for each value in the `key` column. Calling `merge`
with these objects we obtain:

In [27]:
pd.merge(df1, df2)

,key,data1,data2
0,b,0,1
1,b,1,1
2,b,6,1
3,a,2,0
4,a,4,0
5,a,5,0


#overlap <br/>
Note that I didn’t specify which column to join on. If that information is not specified,
`merge` uses the **overlapping column names as the keys**. It’s a good practice to
specify explicitly, though:

In [28]:
pd.merge(df1, df2, on='key')

,key,data1,data2
0,b,0,1
1,b,1,1
2,b,6,1
3,a,2,0
4,a,4,0
5,a,5,0


If the column names are different in each object:

In [29]:
df3 = pd.DataFrame({'lkey': ['b', 'b', 'a', 'c', 'a', 'a', 'b'], # key เหมือน df1
                    'data1': range(7)})
df4 = pd.DataFrame({'rkey': ['a', 'b', 'd'], # key เหมือน df2
                    'data2': range(3)})
pd.merge(df3, df4, left_on='lkey', right_on='rkey')

,lkey,data1,rkey,data2
0,b,0,b,1
1,b,1,b,1
2,b,6,b,1
3,a,2,a,0
4,a,4,a,0
5,a,5,a,0


#inner #default <br/>
You may notice that the 'c' and 'd' values and associated data are missing from the
result. By default `merge` does an 'inner' join; the keys in the result are the **intersection**,
or the common set found in both tables.

#outer <br/>
สำหรับ outer join:

ที่ index 'c' มี data1 = 3 ตามที่สร้าง df1
```
 key  data1
0   b      0
1   b      1
2   a      2
3   c      3
4   a      4
5   a      5
6   b      6
```

ที่ index 'd' มี data2 = 2 ตามที่สร้าง df2
```
  key  data2
0   a      0
1   b      1
2   d      2
```

The outer join takes the **union of the keys**, combining the
effect of applying both left and right joins:

In [30]:
pd.merge(df1, df2, how='outer')

,key,data1,data2
0,b,0.0,1.0
1,b,1.0,1.0
2,b,6.0,1.0
3,a,2.0,0.0
4,a,4.0,0.0
5,a,5.0,0.0
6,c,3.0,NaN
7,d,NaN,2.0


Many-to-many merges:

> Many-to-many joins form the Cartesian product of the rows. Since there were three
'b' rows in the left DataFrame and two in the right one, there are six 'b' rows in the
result.


In [31]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'],
                    'data1': range(6)})
df2 = pd.DataFrame({'key': ['a', 'b', 'a', 'b', 'd'],
                    'data2': range(5)})
print(df1)
print(df2)
pd.merge(df1, df2, on='key', how='left')
#observation
# key และ data1 ใน table ผลลัพธ์ เรียงเหมือน ใน df1 
# ยกเว้นว่ามันมีการ repeat ซ้ำๆเพื่อจับกับ data2 ที่แตกต่างกัน (เหมือนตอนสั่ง rep(..., each=x) ใน R)

  key  data1
0   b      0
1   b      1
2   a      2
3   c      3
4   a      4
5   b      5
  key  data2
0   a      0
1   b      1
2   a      2
3   b      3
4   d      4


,key,data1,data2
0,b,0,1.0
1,b,0,3.0
2,b,1,1.0
3,b,1,3.0
4,a,2,0.0
5,a,2,2.0
6,c,3,NaN
7,a,4,0.0
8,a,4,2.0
9,b,5,1.0


In [32]:
pd.merge(df1, df2, how='inner')
#observation
# key และ data1 ใน table ผลลัพธ์ เรียง ตาม key b ให้หมดก่อน แล้วค่อยตามด้วย key a
# data1 หนึ่งค่า จะไปจับกับ data2 ทีละตัวจนครบ โดยเรียงออกมาเป็น rows ที่อยู่ติดกัน

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,b,5,1
5,b,5,3
6,a,2,0
7,a,2,2
8,a,4,0
9,a,4,2


In [33]:
left = pd.DataFrame({'key1': ['foo', 'foo', 'bar'],
                     'key2': ['one', 'two', 'one'],
                     'lval': [1, 2, 3]})
right = pd.DataFrame({'key1': ['foo', 'foo', 'bar', 'bar'],
                      'key2': ['one', 'one', 'one', 'two'],
                      'rval': [4, 5, 6, 7]})
pd.merge(left, right, on=['key1', 'key2'], how='outer')

,key1,key2,lval,rval
0,foo,one,1.0,4.0
1,foo,one,1.0,5.0
2,foo,two,2.0,NaN
3,bar,one,3.0,6.0
4,bar,two,NaN,7.0


#ลองเอง # ไปดูใน 4experiment.ipynb
```
left = pd.DataFrame({'lkey1': ['foo', 'foo', 'bar'],
                     'lkey2': ['one', 'two', 'one'],
                     'lval': [1, 2, 3]})
right = pd.DataFrame({'rkey1': ['foo', 'foo', 'bar', 'bar'],
                      'rkey2': ['one', 'one', 'one', 'two'],
                      'rval': [4, 5, 6, 7]})
pd.merge(left, right, left_on=['lkey1', 'lkey2'], right_on=['rkey1', 'rkey2'], how='outer')
```

In [34]:
pd.merge(left, right, on='key1')

,key1,key2_x,lval,key2_y,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


In [35]:
pd.merge(left, right, on='key1', suffixes=('_left', '_right'))

,key1,key2_left,lval,key2_right,rval
0,foo,one,1,one,4
1,foo,one,1,one,5
2,foo,two,2,one,4
3,foo,two,2,one,5
4,bar,one,3,one,6
5,bar,one,3,two,7


### Merging on Index

In [36]:
left1 = pd.DataFrame({'key': ['a', 'b', 'a', 'a', 'b', 'c'],
                      'value': range(6)})
right1 = pd.DataFrame({'group_val': [3.5, 7]}, index=['a', 'b'])
left1
right1
pd.merge(left1, right1, left_on='key', right_index=True)

,key,value,group_val
0,a,0,3.5
2,a,2,3.5
3,a,3,3.5
1,b,1,7.0
4,b,4,7.0


In [37]:
pd.merge(left1, right1, left_on='key', right_index=True, how='outer')

,key,value,group_val
0,a,0,3.5
2,a,2,3.5
3,a,3,3.5
1,b,1,7.0
4,b,4,7.0
5,c,5,NaN


In [38]:
lefth = pd.DataFrame({'key1': ['Ohio', 'Ohio', 'Ohio',
                               'Nevada', 'Nevada'],
                      'key2': [2000, 2001, 2002, 2001, 2002],
                      'data': np.arange(5.)})
righth = pd.DataFrame(np.arange(12).reshape((6, 2)),
                      index=[['Nevada', 'Nevada', 'Ohio', 'Ohio',
                              'Ohio', 'Ohio'],
                             [2001, 2000, 2000, 2000, 2001, 2002]],
                      columns=['event1', 'event2'])
print(lefth)
print(righth)

     key1  key2  data
0    Ohio  2000   0.0
1    Ohio  2001   1.0
2    Ohio  2002   2.0
3  Nevada  2001   3.0
4  Nevada  2002   4.0
             event1  event2
Nevada 2001       0       1
       2000       2       3
Ohio   2000       4       5
       2000       6       7
       2001       8       9
       2002      10      11


In [39]:
pd.merge(lefth, righth, left_on=['key1', 'key2'], right_index=True)
pd.merge(lefth, righth, left_on=['key1', 'key2'],
         right_index=True, how='outer')
# Nev 2001 -> มีทั้งใน lefth และ righth, Nev 2002 มีแต่ lefth, Nev 2000 มีแต่ righth

,key1,key2,data,event1,event2
0,Ohio,2000,0.0,4.0,5.0
0,Ohio,2000,0.0,6.0,7.0
1,Ohio,2001,1.0,8.0,9.0
2,Ohio,2002,2.0,10.0,11.0
3,Nevada,2001,3.0,0.0,1.0
4,Nevada,2002,4.0,NaN,NaN
4,Nevada,2000,NaN,2.0,3.0


In [40]:
left2 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]],
                     index=['a', 'c', 'e'],
                     columns=['Ohio', 'Nevada'])
right2 = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]],
                      index=['b', 'c', 'd', 'e'],
                      columns=['Missouri', 'Alabama'])
left2
right2
pd.merge(left2, right2, how='outer', left_index=True, right_index=True)
# common index คือ c, e ดังนั้น row ที่มี Nan คือ a, b and d

,Ohio,Nevada,Missouri,Alabama
a,1.0,2.0,NaN,NaN
b,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0
d,NaN,NaN,11.0,12.0
e,5.0,6.0,13.0,14.0


DataFrame has a convenient join instance for **merging by index**. It can also be used
to combine together many DataFrame objects having the same or similar indexes(1) but
non-overlapping columns(2). In the prior example, we could have written:(3)

1.  left2, right2 มี similar index ซึ่งถูกเอามาใช้ในการ merge
2.  เหมือนที่อธิบายไว้ใน #outer ก็คือ if not specified การ merge จะใช้ keys เป็น columns ที่ overlap ดังนั้นในกรณีของ non-overlapping columns จะไม่สามารถทำแบบนี้ได้
3. ประโยคนี้ imply ว่าในบรรทัดนี้ ผล่ลัพธ์จะเหมือนก่อนหน้า

In [43]:
left2.join(right2, how='outer')

,Ohio,Nevada,Missouri,Alabama
a,1.0,2.0,NaN,NaN
b,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0
d,NaN,NaN,11.0,12.0
e,5.0,6.0,13.0,14.0


In part for legacy reasons (i.e., much earlier versions of pandas), DataFrame’s join
method performs a left join on the join keys, exactly preserving the left frame’s row
index. It also supports joining the index of the passed DataFrame on one of the columns
of the calling DataFrame:

The calling DataFrame: left1 (with column `key` being joined on) <br/>
The passed DataFrame: right1 (with its index being joined on)

Set of keys of the result DataFrame is the same as left1's (left join is performed)
```
left1 = pd.DataFrame({'key': ['a', 'b', 'a', 'a', 'b', 'c'],
                      'value': range(6)})
right1 = pd.DataFrame({'group_val': [3.5, 7]}, index=['a', 'b'])
```

In [44]:
left1.join(right1, on='key')

,key,value,group_val
0,a,0,3.5
1,b,1,7.0
2,a,2,3.5
3,a,3,3.5
4,b,4,7.0
5,c,5,NaN


Lastly, for simple index-on-index merges, you can pass a list of DataFrames to `join` as
an alternative to using the more general `concat` function described in the next
section:

In [45]:
another = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [16., 17.]],
                       index=['a', 'c', 'e', 'f'],
                       columns=['New York', 'Oregon'])
another

,New York,Oregon
a,7.0,8.0
c,9.0,10.0
e,11.0,12.0
f,16.0,17.0


```
left2 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]],
                     index=['a', 'c', 'e'],
                     columns=['Ohio', 'Nevada'])
right2 = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]],
                      index=['b', 'c', 'd', 'e'],
                      columns=['Missouri', 'Alabama'])
```

In [46]:
left2.join([right2, another]) # left join. keys come from left2 only

,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1.0,2.0,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0,9.0,10.0
e,5.0,6.0,13.0,14.0,11.0,12.0


In [48]:
left2.join([right2, another], how='outer')
# a, c, e --> left2
# b, d --> right2
# f --> another

,Ohio,Nevada,Missouri,Alabama,New York,Oregon
a,1.0,2.0,NaN,NaN,7.0,8.0
c,3.0,4.0,9.0,10.0,9.0,10.0
e,5.0,6.0,13.0,14.0,11.0,12.0
b,NaN,NaN,7.0,8.0,NaN,NaN
d,NaN,NaN,11.0,12.0,NaN,NaN
f,NaN,NaN,NaN,NaN,16.0,17.0


### Concatenating Along an Axis

In [49]:
arr = np.arange(12).reshape((3, 4))
arr
np.concatenate([arr, arr], axis=1)

array([[ 0,  1,  2,  3,  0,  1,  2,  3],
       [ 4,  5,  6,  7,  4,  5,  6,  7],
       [ 8,  9, 10, 11,  8,  9, 10, 11]])

In [50]:
s1 = pd.Series([0, 1], index=['a', 'b'])
s2 = pd.Series([2, 3, 4], index=['c', 'd', 'e'])
s3 = pd.Series([5, 6], index=['f', 'g'])

In [51]:
pd.concat([s1, s2, s3])

a    0
b    1
c    2
d    3
e    4
f    5
g    6
dtype: int64

In [52]:
#no_keys
pd.concat([s1, s2, s3], axis=1)

,0,1,2
a,0.0,NaN,NaN
b,1.0,NaN,NaN
c,NaN,2.0,NaN
d,NaN,3.0,NaN
e,NaN,4.0,NaN
f,NaN,NaN,5.0
g,NaN,NaN,6.0


In this case there is no overlap on the other axis, which as you can see is the sorted
union (the 'outer' join) of the indexes. You can instead intersect them by passing
join='inner':

In [54]:
s4 = pd.concat([s1, s3])
print(s4, '\n')
print(pd.concat([s1, s4], axis=1), '\n')
print(pd.concat([s1, s4], axis=1, join='inner'))

a    0
b    1
f    5
g    6
dtype: int64 

     0  1
a  0.0  0
b  1.0  1
f  NaN  5
g  NaN  6 

   0  1
a  0  0
b  1  1


A potential issue is that the concatenated pieces are not identifiable in the result. Suppose
instead you wanted to create a hierarchical index on the concatenation axis. To
do this, use the `keys` argument:

In [57]:
result = pd.concat([s1, s1, s3], keys=['one', 'two', 'three'])
print(result)
result.unstack()

one    a    0
       b    1
two    a    0
       b    1
three  f    5
       g    6
dtype: int64


,a,b,f,g
one,0.0,1.0,NaN,NaN
two,0.0,1.0,NaN,NaN
three,NaN,NaN,5.0,6.0


In [69]:
#ลองเอง # ลองใช้ดาต้าชุดเดียวกันกับ cell ถัดไปเพื่อเปรียบเทียบ side-by-side
r=pd.concat([s1, s2, s3], keys=['one', 'two', 'three'])
print(r)
r.unstack()

one    a    0
       b    1
two    c    2
       d    3
       e    4
three  f    5
       g    6
dtype: int64


,a,b,c,d,e,f,g
one,0.0,1.0,NaN,NaN,NaN,NaN,NaN
two,NaN,NaN,2.0,3.0,4.0,NaN,NaN
three,NaN,NaN,NaN,NaN,NaN,5.0,6.0


In the case of combining Series along `axis=1`, the `keys` become the DataFrame column headers

In [58]:
pd.concat([s1, s2, s3], axis=1, keys=['one', 'two', 'three'])
# อย่าสับสน ถึงไม่ใส่ keys ก็ได้ data frame แบบเดียวกัน ... ดู #no_keys

#transpose
# ใน column one จะเป็น data จาก s1; column two จาก s2
# ส่วนข้างบน ใน row one จะมาจาก s1; row two จาก s2

,one,two,three
a,0.0,NaN,NaN
b,1.0,NaN,NaN
c,NaN,2.0,NaN
d,NaN,3.0,NaN
e,NaN,4.0,NaN
f,NaN,NaN,5.0
g,NaN,NaN,6.0


In [124]:
df1 = pd.DataFrame(np.arange(6).reshape(3, 2), index=['a', 'b', 'c'],
                   columns=['one', 'two'])
df2 = pd.DataFrame(5 + np.arange(4).reshape(2, 2), index=['a', 'c'],
                   columns=['three', 'four'])
print(df1)
print(df2)
pd.concat([df1, df2], axis=1, keys=['level1', 'level2'])

   one  two
a    0    1
b    2    3
c    4    5
   three  four
a      5     6
c      7     8


level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

In [60]:
pd.concat({'level1': df1, 'level2': df2}, axis=1) # เหมือนกัน

level1     level2     
     one two  three four
a      0   1    5.0  6.0
b      2   3    NaN  NaN
c      4   5    7.0  8.0

In [127]:
x = pd.concat([df1, df2], axis=1, keys=['level1', 'level2'],
          names=['upper', 'lower'])
print(x.columns.names)
x

['upper', 'lower']


upper level1     level2     
lower    one two  three four
a          0   1    5.0  6.0
b          2   3    NaN  NaN
c          4   5    7.0  8.0

In [63]:
df1 = pd.DataFrame(np.random.randn(3, 4), columns=['a', 'b', 'c', 'd'])
df2 = pd.DataFrame(np.random.randn(2, 3), columns=['b', 'd', 'a'])
print(df1)
print(df2)

          a         b         c         d
0  0.523772  0.000940  1.343810 -0.713544
1 -0.831154 -2.370232 -1.860761 -0.860757
2  0.560145 -1.265934  0.119827 -1.063512
          b         d         a
0  0.332883 -2.359419 -0.199543
1 -1.541996 -0.970736 -1.307030


In [64]:
pd.concat([df1, df2], ignore_index=True) # แค่ทำให้ index ไม่เป็น 0,1,2,0,1 ซึ่ง 0,1 เป็น index ของ df2

,a,b,c,d
0,0.523772,0.000940,1.343810,-0.713544
1,-0.831154,-2.370232,-1.860761,-0.860757
2,0.560145,-1.265934,0.119827,-1.063512
3,-0.199543,0.332883,NaN,-2.359419
4,-1.307030,-1.541996,NaN,-0.970736


In [65]:
#ลองเอง
pd.concat([df1, df2])

,a,b,c,d
0,0.523772,0.000940,1.343810,-0.713544
1,-0.831154,-2.370232,-1.860761,-0.860757
2,0.560145,-1.265934,0.119827,-1.063512
0,-0.199543,0.332883,NaN,-2.359419
1,-1.307030,-1.541996,NaN,-0.970736


In [66]:
#ลองเอง
pd.concat([df1, df2], axis=0)

,a,b,c,d
0,0.523772,0.000940,1.343810,-0.713544
1,-0.831154,-2.370232,-1.860761,-0.860757
2,0.560145,-1.265934,0.119827,-1.063512
0,-0.199543,0.332883,NaN,-2.359419
1,-1.307030,-1.541996,NaN,-0.970736


### Combining Data with Overlap

In [70]:
a = pd.Series([np.nan, 2.5, np.nan, 3.5, 4.5, np.nan],
              index=['f', 'e', 'd', 'c', 'b', 'a'])
b = pd.Series(np.arange(len(a), dtype=np.float64),
              index=['f', 'e', 'd', 'c', 'b', 'a'])
b[-1] = np.nan
a
b
np.where(pd.isnull(a), b, a)

array([0. , 2.5, 2. , 3.5, 4.5, nan])

In [98]:
#ลองเอง
print(f"b[:-2]\n{b[:-2].sort_index()}", '\n')
print(f"a[2:]\n{a[2:].sort_index()}")

b[:-2]
c    3.0
d    2.0
e    1.0
f    0.0
dtype: float64 

a[2:]
a    NaN
b    4.5
c    3.5
d    NaN
dtype: float64


In [74]:
b[:-2].combine_first(a[2:])

a    NaN
b    4.5
c    3.0
d    2.0
e    1.0
f    0.0
dtype: float64

In [96]:
#ลองเอง 
# สรุปได้ว่า: ถ้า index ใน bb ไม่มี หรือมีแต่ value เป้น NaN จะเอาค่าจาก a[2:]

bb = b[:-2].copy().sort_index()
bb['c'] = np.nan
print(bb)

bb.combine_first(a[2:])
# index 'a' is missing
# 'b' is missing
# 'c' value is NaN
# 'd', 'e', 'f' เอามาจาก bb

c    NaN
d    2.0
e    1.0
f    0.0
dtype: float64


a    NaN
b    4.5
c    3.5
d    2.0
e    1.0
f    0.0
dtype: float64

In [97]:
df1 = pd.DataFrame({'a': [1., np.nan, 5., np.nan],
                    'b': [np.nan, 2., np.nan, 6.],
                    'c': range(2, 18, 4)})
df2 = pd.DataFrame({'a': [5., 4., np.nan, 3., 7.],
                    'b': [np.nan, 3., 4., 6., 8.]})
df1
df2
df1.combine_first(df2)

,a,b,c
0,1.0,NaN,2.0
1,4.0,2.0,6.0
2,5.0,4.0,10.0
3,3.0,6.0,14.0
4,7.0,8.0,NaN


ที่ตำแหน่ง 4-c เป็น NaN ด้วยคนละเหตุผลกับตัวอย่าง Series

ในกรณี Series, index 'a' is missing from `b[:-2]`แต่ไม่ missing from `a[2:]` (ถึงแม้ค่าจะเป็น NaN)

ในกรณี DataFrame นี้ 4-c missing from `df1` (row สูงสุดคือ row ที่ 4 with index==3) และ from `df2` (ไม่มี column 'c')

แต่ว่า row '4' จำเป็นต้องมี เพราะในคอลัมน์อื่นของ df2 มี rowนี้ และ column 'c' ก็จำเป็นต้องมีเพราะมาจาก `df1`

## Reshaping and Pivoting

### Reshaping with Hierarchical Indexing

```
stack
    This “rotates” or pivots from the columns in the data to the rows
unstack
    This pivots from the rows into the columns
```

In [111]:
data = pd.DataFrame(np.arange(6).reshape((2, 3)),
                    index=pd.Index(['Ohio', 'Colorado'], name='state'),
                    columns=pd.Index(['one', 'two', 'three'],
                    name='number'))
data

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


Using the `stack` method on this data pivots the columns into the rows, producing a
Series:

In [112]:
result = data.stack()
result

state     number
Ohio      one       0
          two       1
          three     2
Colorado  one       3
          two       4
          three     5
dtype: int32

In [113]:
result.unstack()

number,one,two,three
state,,,
Ohio,0,1,2
Colorado,3,4,5


By default the innermost level is unstacked (same with stack). You can unstack a different
level by passing a level number or name:

In [114]:
result.unstack(0)
result.unstack('state')

state,Ohio,Colorado
number,,
one,0,3
two,1,4
three,2,5


In [115]:
#ลองเอง #transpose #union
# When unstacking the outer level, there may be a situation
# that labels in the inner level in each of the subgroup are different.
# คาดว่า final index จะเป็น union ของ labels จากทุกๆ subgroup
# นอกจากนี้ เชื่อว่า r.unstack() กับ r.unstack(0) จะเป็น transpose ของกันและกัน

ss2 = pd.concat([s2, pd.Series({'a': -1})])

r=pd.concat([s1, ss2, s3], keys=['one', 'two', 'three'])
print(r, '\n')
print(r.unstack(), '\n')

print(r.unstack(0))
# expect this: 
# row a,b มี non-NaN ที่ column one, two
# row c, d, e แค่ col two
# row f,g แค่ col three

one    a    0
       b    1
two    c    2
       d    3
       e    4
       a   -1
three  f    5
       g    6
dtype: int64 

         a    b    c    d    e    f    g
one    0.0  1.0  NaN  NaN  NaN  NaN  NaN
two   -1.0  NaN  2.0  3.0  4.0  NaN  NaN
three  NaN  NaN  NaN  NaN  NaN  5.0  6.0 

   one  two  three
a  0.0 -1.0    NaN
b  1.0  NaN    NaN
c  NaN  2.0    NaN
d  NaN  3.0    NaN
e  NaN  4.0    NaN
f  NaN  NaN    5.0
g  NaN  NaN    6.0


Unstacking might introduce missing data if all of the values in the level aren’t found
in each of the subgroups:

In [122]:
s1 = pd.Series([0, 1, 2, 3], index=['a', 'b', 'c', 'd'])
s2 = pd.Series([4, 5, 6], index=['c', 'd', 'e'])
data2 = pd.concat([s1, s2], keys=['one', 'two'])
print(data2)
data2.unstack()

one  a    0
     b    1
     c    2
     d    3
two  c    4
     d    5
     e    6
dtype: int64


,a,b,c,d,e
one,0.0,1.0,2.0,3.0,NaN
two,NaN,NaN,4.0,5.0,6.0


Stacking filters out missing data by default, so the operation is more easily invertible:

In [134]:
data2.unstack().stack()

one  a    0.0
     b    1.0
     c    2.0
     d    3.0
two  c    4.0
     d    5.0
     e    6.0
dtype: float64

In [130]:
data2.unstack().stack(dropna=False)

one  a    0.0
     b    1.0
     c    2.0
     d    3.0
     e    NaN
two  a    NaN
     b    NaN
     c    4.0
     d    5.0
     e    6.0
dtype: float64

When you unstack in a DataFrame, the level unstacked becomes the lowest level in
the result:

In [176]:
#lowest #innermost
df = pd.DataFrame({'left': result, 'right': result + 5},
                  columns=pd.Index(['left', 'right'], name='side'))
print(df)
df.unstack('state')

side             left  right
state    number             
Ohio     one        0      5
         two        1      6
         three      2      7
Colorado one        3      8
         two        4      9
         three      5     10


side   left          right         
state  Ohio Colorado  Ohio Colorado
number                             
one       0        3     5        8
two       1        4     6        9
three     2        5     7       10

In [128]:
#ลองเอง # same result
pd.concat([df['left'].unstack('state'), df['right'].unstack('state')], axis=1, keys=['ซ้าย','ขวา'], names=['side', 'state'])

side   ซ้าย           ขวา         
state  Ohio Colorado Ohio Colorado
number                            
one       0        3    5        8
two       1        4    6        9
three     2        5    7       10

In [ ]:
df.unstack('state').stack('side')

### Pivoting “Long” to “Wide” Format

In [162]:
data = pd.read_csv('examples/macrodata.csv')
data.head()
periods = pd.PeriodIndex(year=data.year, quarter=data.quarter,
                         name='date')
columns = pd.Index(['realgdp', 'infl', 'unemp'], name='item')
data = data.reindex(columns=columns)
data.index = periods.to_timestamp('D', 'end')
ldata = data.stack().reset_index().rename(columns={0: 'value'})
# ตอน stack ชื่อคอลัมน์สามตัวถูกเอามา mixed อยู่ใน index เดียวกัน หลังจากนั้นพอ reset_index ก็จะกลายเป็นคอลัมน์เดียวที่มีทั้ง realgdp, infl และ unemp
#stack=mix

In [163]:
#scratchpad # ตอน reset_index ชื่อ index จะกลายเป็นชื่อ column
d = [0, 1, 2, 3]; d.extend([4, 5, 6])
inner = ['a', 'b', 'c', 'd']; inner.extend(['c', 'd', 'e'])
outer = ['one']*4; outer.extend(['two']*3)

example = pd.Series(d, index=pd.MultiIndex.from_arrays([outer, inner],
                       names=['number', 'letter']))
print(example, '\n')
print(example.reset_index())

number  letter
one     a         0
        b         1
        c         2
        d         3
two     c         4
        d         5
        e         6
dtype: int64 

  number letter  0
0    one      a  0
1    one      b  1
2    one      c  2
3    one      d  3
4    two      c  4
5    two      d  5
6    two      e  6


In [164]:
ldata[:10]

,date,item,value
0,1959-03-31 23:59:59.999999999,realgdp,2710.349
1,1959-03-31 23:59:59.999999999,infl,0.000
2,1959-03-31 23:59:59.999999999,unemp,5.800
3,1959-06-30 23:59:59.999999999,realgdp,2778.801
4,1959-06-30 23:59:59.999999999,infl,2.340
5,1959-06-30 23:59:59.999999999,unemp,5.100
6,1959-09-30 23:59:59.999999999,realgdp,2775.488
7,1959-09-30 23:59:59.999999999,infl,2.740
8,1959-09-30 23:59:59.999999999,unemp,5.300
9,1959-12-31 23:59:59.999999999,realgdp,2785.204


The first two values passed are the columns to be used respectively as the row and
column index, then finally an optional value column to fill the DataFrame.

In [165]:
pivoted = ldata.pivot('date', 'item', 'value')
pivoted

item,infl,realgdp,unemp
date,,,
1959-03-31 23:59:59.999999999,0.00,2710.349,5.8
1959-06-30 23:59:59.999999999,2.34,2778.801,5.1
1959-09-30 23:59:59.999999999,2.74,2775.488,5.3
1959-12-31 23:59:59.999999999,0.27,2785.204,5.6
1960-03-31 23:59:59.999999999,2.31,2847.699,5.2
...,...,...,...
2008-09-30 23:59:59.999999999,-3.16,13324.600,6.0
2008-12-31 23:59:59.999999999,-8.79,13141.920,6.9
2009-03-31 23:59:59.999999999,0.94,12925.410,8.1


In [169]:
#ลองเอง 
# ได้ไอเดียมาจาก #unstack=unmix
ldata.set_index(['date', 'item']).unstack('item')['value']

item,infl,realgdp,unemp
date,,,
1959-03-31 23:59:59.999999999,0.00,2710.349,5.8
1959-06-30 23:59:59.999999999,2.34,2778.801,5.1
1959-09-30 23:59:59.999999999,2.74,2775.488,5.3
1959-12-31 23:59:59.999999999,0.27,2785.204,5.6
1960-03-31 23:59:59.999999999,2.31,2847.699,5.2
...,...,...,...
2008-09-30 23:59:59.999999999,-3.16,13324.600,6.0
2008-12-31 23:59:59.999999999,-8.79,13141.920,6.9
2009-03-31 23:59:59.999999999,0.94,12925.410,8.1


In [160]:
ldata['value2'] = np.random.randn(len(ldata))
ldata[:10]

,date,item,value,value2
0,1959-03-31 23:59:59.999999999,realgdp,2710.349,-0.897609
1,1959-03-31 23:59:59.999999999,infl,0.000,1.844805
2,1959-03-31 23:59:59.999999999,unemp,5.800,1.253168
3,1959-06-30 23:59:59.999999999,realgdp,2778.801,-1.490932
4,1959-06-30 23:59:59.999999999,infl,2.340,-0.027734
5,1959-06-30 23:59:59.999999999,unemp,5.100,1.375236
6,1959-09-30 23:59:59.999999999,realgdp,2775.488,-0.025208
7,1959-09-30 23:59:59.999999999,infl,2.740,-0.667880
8,1959-09-30 23:59:59.999999999,unemp,5.300,-2.868018
9,1959-12-31 23:59:59.999999999,realgdp,2785.204,0.210689


By omitting the last argument, you obtain a DataFrame with hierarchical columns:

In [154]:
pivoted = ldata.pivot('date', 'item')
print(pivoted[:5])
pivoted['value'][:5]

                              value                    value2            \
item                           infl   realgdp unemp      infl   realgdp   
date                                                                      
1959-03-31 23:59:59.999999999  0.00  2710.349   5.8  0.377984  0.286350   
1959-06-30 23:59:59.999999999  2.34  2778.801   5.1  1.349742  0.331286   
1959-09-30 23:59:59.999999999  2.74  2775.488   5.3 -0.011862  0.246674   
1959-12-31 23:59:59.999999999  0.27  2785.204   5.6 -0.919262  1.327195   
1960-03-31 23:59:59.999999999  2.31  2847.699   5.2  0.758363  0.022185   

                                         
item                              unemp  
date                                     
1959-03-31 23:59:59.999999999 -0.753887  
1959-06-30 23:59:59.999999999  0.069877  
1959-09-30 23:59:59.999999999  1.004812  
1959-12-31 23:59:59.999999999 -1.549106  
1960-03-31 23:59:59.999999999 -0.660524  


item,infl,realgdp,unemp
date,,,
1959-03-31 23:59:59.999999999,0.00,2710.349,5.8
1959-06-30 23:59:59.999999999,2.34,2778.801,5.1
1959-09-30 23:59:59.999999999,2.74,2775.488,5.3
1959-12-31 23:59:59.999999999,0.27,2785.204,5.6
1960-03-31 23:59:59.999999999,2.31,2847.699,5.2


In [156]:
#unstack=unmix
unstacked = ldata.set_index(['date', 'item']).unstack('item')
print(ldata.set_index(['date', 'item'])[:7]) # ไอ้ตัวนี้เมื่อสั่ง unstack ก็จะได้ unstacked
unstacked[:7]
#recall #stack=mix 
# ถ้า stack จะทำให้ชื่อคอลัมน์มา mixed กันใน index
# กลับกัน ถ้า unstack ก็จะทำให้ index ซึ่งเก็บค่าที่ mix กัน กลายเป็นหลายๆคอลัมน์แทน

                                          value    value2
date                          item                       
1959-03-31 23:59:59.999999999 realgdp  2710.349  0.286350
                              infl        0.000  0.377984
                              unemp       5.800 -0.753887
1959-06-30 23:59:59.999999999 realgdp  2778.801  0.331286
                              infl        2.340  1.349742
                              unemp       5.100  0.069877
1959-09-30 23:59:59.999999999 realgdp  2775.488  0.246674


value                    value2            \
item                           infl   realgdp unemp      infl   realgdp   
date                                                                      
1959-03-31 23:59:59.999999999  0.00  2710.349   5.8  0.377984  0.286350   
1959-06-30 23:59:59.999999999  2.34  2778.801   5.1  1.349742  0.331286   
1959-09-30 23:59:59.999999999  2.74  2775.488   5.3 -0.011862  0.246674   
1959-12-31 23:59:59.999999999  0.27  2785.204   5.6 -0.919262  1.327195   
1960-03-31 23:59:59.999999999  2.31  2847.699   5.2  0.758363  0.022185   
1960-06-30 23:59:59.999999999  0.14  2834.390   5.2 -0.010032  0.862580   
1960-09-30 23:59:59.999999999  2.70  2839.022   5.6  0.852965  0.670216   

                                         
item                              unemp  
date                                     
1959-03-31 23:59:59.999999999 -0.753887  
1959-06-30 23:59:59.999999999  0.069877  
1959-09-30 23:59:59.999999999  1.004812  
1959-12-31 23:59:59.999999999 -1.549106  
1960-03-31 23:59:59.999999999 -0.660524  
1960-06-30 23:59:59.999999999  0.050009  
1960-09-30 23:59:59.999999999 -0.955869

### Pivoting “Wide” to “Long” Format

In [204]:
df = pd.DataFrame({'key': ['foo', 'bar', 'baz'],
                   'A': [1, 2, 3],
                   'B': [4, 5, 6],
                   'C': [7, 8, 9]})
df

,key,A,B,C
0,foo,1,4,7
1,bar,2,5,8
2,baz,3,6,9


In [203]:
melted = pd.melt(df, ['key'])
melted

,key,variable,value
0,foo,A,1.0
1,bar,A,2.0
2,baz,A,3.0
3,foo,B,4.0
4,bar,B,5.0
5,baz,B,6.0
6,foo,C,7.0
7,bar,C,8.0
8,baz,C,NaN


In [179]:
reshaped = melted.pivot('key', 'variable', 'value')
reshaped

variable,A,B,C
key,,,
bar,2,5,8
baz,3,6,9
foo,1,4,7


In [180]:
reshaped.reset_index()

variable,key,A,B,C
0,bar,2,5,8
1,baz,3,6,9
2,foo,1,4,7


In [190]:
#ลองเอง #invert
print(df,'\n')
print(pd.melt(df, ['key']).pivot('key', 'variable', 'value').reset_index(),'\n')
print(df.set_index('key').stack().unstack().reset_index()) # ถ้ามาทาง route นี้จะไม่มีการตั้งชื่อ 'variable'

   key  A  B  C
0  foo  1  4  7
1  bar  2  5  8
2  baz  3  6  9 

variable  key  A  B  C
0         bar  2  5  8
1         baz  3  6  9
2         foo  1  4  7 

   key  A  B  C
0  foo  1  4  7
1  bar  2  5  8
2  baz  3  6  9


In [201]:
#ลองเอง # คนละคำถาม คือว่าถ้า ถ้า key มีค่าซ้ำกันจะเป็นยังไง
df_2 = pd.DataFrame({'key': ['foo', 'bar', 'foo'],
                   'A': [1, 2, 3],
                   'B': [4, 5, 6],
                   'C': [7, 8, 9]})
print(df_2)

pd.melt(df_2, 'key').sort_values(by=['key','value'], ascending=[False, True])


   key  A  B  C
0  foo  1  4  7
1  bar  2  5  8
2  foo  3  6  9


,key,variable,value
0,foo,A,1
2,foo,A,3
3,foo,B,4
5,foo,B,6
6,foo,C,7
8,foo,C,9
1,bar,A,2
4,bar,B,5
7,bar,C,8


In [182]:
pd.melt(df, id_vars=['key'], value_vars=['A', 'B']) # Rows with variable=='C' are discarded

,key,variable,value
0,foo,A,1
1,bar,A,2
2,baz,A,3
3,foo,B,4
4,bar,B,5
5,baz,B,6


In [191]:
pd.melt(df, value_vars=['A', 'B', 'C'])

,variable,value
0,A,1
1,A,2
2,A,3
3,B,4
4,B,5
5,B,6
6,C,7
7,C,8
8,C,9


In [208]:
print(df_2)
pd.melt(df_2, value_vars=['key', 'A', 'B'])
# ระหว่าง label ที่ถูกเอามา เป็น value_vars ด้วยกันจะไม่มีความสัมพันธ์ต่อกัน 
# ไม่เหมือนกับ ที่มันมีความสัมพันธ์ กับ id_vars: เวลา melt A จะมาปรากฎ ที่ row ที่ key เป็น foo สองครั้ง (ดู cell ถัดไป)

   key  A  B  C
0  foo  1  4  7
1  bar  2  5  8
2  foo  3  6  9


,variable,value
0,key,foo
1,key,bar
2,key,foo
3,A,1
4,A,2
5,A,3
6,B,4
7,B,5
8,B,6


In [210]:
#ลองเอง ต่อจากข้างบน
pd.melt(df_2, 'key', value_vars=['A', 'B'])

,key,variable,value
0,foo,A,1
1,bar,A,2
2,foo,A,3
3,foo,B,4
4,bar,B,5
5,foo,B,6


In [217]:
#ลองเอง # ถ้ามี NaN จะเป็นอย่างไร
df_3 = pd.DataFrame({'key': ['foo', 'bar', 'foo'],
                   'A': [1, np.nan, np.nan],
                   'B': [np.nan, 5, np.nan],
                   'C': [np.nan, np.nan, 9]})
print(df_3)

pd.melt(df_3, 'key')

   key    A    B    C
0  foo  1.0  NaN  NaN
1  bar  NaN  5.0  NaN
2  foo  NaN  NaN  9.0


,key,variable,value
0,foo,A,1.0
1,bar,A,NaN
2,foo,A,NaN
3,foo,B,NaN
4,bar,B,5.0
5,foo,B,NaN
6,foo,C,NaN
7,bar,C,NaN
8,foo,C,9.0


In [218]:
#ลองเอง ต่อจากข้างบน
df_3.set_index(['key']).stack()

key   
foo  A    1.0
bar  B    5.0
foo  C    9.0
dtype: float64

## Conclusion